In [1]:
!pip install -q onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 8.9 MB/s eta 0:00:00


In [2]:
import os, time, platform, subprocess
from itertools import cycle
import numpy as np, cv2, torch, torch.nn as nn
import onnx, onnxruntime as ort

NUM_THREADS = os.cpu_count()          # Colab free tier is usually 2
torch.set_num_threads(NUM_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass                               # already set earlier in this runtime
cv2.setNumThreads(NUM_THREADS)

def cpu_name():
    try:
        for line in subprocess.check_output("lscpu", text=True).splitlines():
            if "Model name" in line:
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "unknown"

print("CPU:", cpu_name())
print("Logical cores:", os.cpu_count())
print(f"Threads: torch intra-op={torch.get_num_threads()}, ORT intra-op={NUM_THREADS}, cv2={cv2.getNumThreads()}")
print(f"torch {torch.__version__} | onnx {onnx.__version__} | onnxruntime {ort.__version__} | opencv {cv2.__version__} | python {platform.python_version()}")

CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
Logical cores: 2
Threads: torch intra-op=2, ORT intra-op=2, cv2=2
torch 2.11.0+cpu | onnx 1.23.2 | onnxruntime 1.31.0 | opencv 5.0.0 | python 3.13.16


In [3]:
# Setup: mount Drive + definitions copied from the notebook
from google.colab import drive
drive.mount('/content/drive')

import os, cv2, numpy as np, pandas as pd, torch.nn as nn

drive_path = '/content/drive/MyDrive/invasive_species_data'
csv_path   = os.path.join(drive_path, 'train_labels.csv')
image_path = os.path.join(drive_path, 'train_images', 'train')

def texture_segmentation(img):
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sobelx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=5)
    sobely = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=5)
    return np.hypot(sobelx, sobely)

def create_model():
    return nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(64, 128, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Linear(86528, 128), nn.ReLU(),
        nn.Linear(128, 1),
        nn.Sigmoid(),
    )

try:
    labels_df = pd.read_csv(csv_path)
    print(f"Labels loaded: {len(labels_df)} rows")
except FileNotFoundError:
    print("train_labels.csv not found, Cell 5 will use synthetic images")

print("model.pth exists:", os.path.exists(os.path.join(drive_path, 'model.pth')))
print("image folder exists:", os.path.exists(image_path))

Mounted at /content/drive
Labels loaded: 2295 rows
model.pth exists: False
image folder exists: True


In [4]:
MODEL_PATH = '/content/drive/MyDrive/invasive_species_data/model.pth'

cpu_model = create_model()
if os.path.exists(MODEL_PATH):
    cpu_model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu', weights_only=True))
    WEIGHTS = "trained"
    print(f"Loaded trained weights from {MODEL_PATH}")
else:
    torch.manual_seed(0)
    WEIGHTS = "random"
    print("WARNING: model.pth not found. Using the SAME architecture with RANDOM weights.")
    print("Latency results are still valid: a dense CNN does the same operations no matter what the weight values are.")
    print("Prediction-related numbers (outputs, int8 agreement) are NOT meaningful with random weights.")
cpu_model.eval()

print(f"Parameters: {sum(p.numel() for p in cpu_model.parameters()):,}")

Latency results are still valid: a dense CNN does the same operations no matter what the weight values are.
Prediction-related numbers (outputs, int8 agreement) are NOT meaningful with random weights.
Parameters: 11,168,513


In [5]:
ONNX_PATH   = '/content/model_fp32.onnx'
LOGITS_PATH = '/content/model_logits.onnx'   # same model minus the Sigmoid, only for the correctness check

# Shares weights with cpu_model. Used because Sigmoid can saturate to 0/1 and hide real differences.
logits_model = nn.Sequential(*list(cpu_model.children())[:-1]).eval()

dummy = torch.randn(1, 1, 224, 224)

def export(m, path):
    kwargs = dict(input_names=['input'], output_names=['output'], opset_version=17,
                  dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})
    try:
        torch.onnx.export(m, dummy, path, dynamo=False, **kwargs)   # classic exporter, honors opset 17
        return "TorchScript exporter (dynamo=False)"
    except TypeError:
        torch.onnx.export(m, dummy, path, **kwargs)                 # older torch without the dynamo arg
        return "default exporter"

with torch.no_grad():
    how = export(cpu_model, ONNX_PATH)
    export(logits_model, LOGITS_PATH)

m = onnx.load(ONNX_PATH)
onnx.checker.check_model(m)
print("Exported with:", how)
print("Opset:", m.opset_import[0].version)
print("Input shape:", [d.dim_param or d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim])
print("Ops in graph:", [n.op_type for n in m.graph.node])
print(f"File size: {os.path.getsize(ONNX_PATH)/1e6:.1f} MB")

/tmp/ipykernel_569/1095670422.py:13: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(m, dummy, path, dynamo=False, **kwargs)   # classic exporter, honors opset 17


Exported with: TorchScript exporter (dynamo=False)
Opset: 17
Input shape: ['batch', 1, 224, 224]
Ops in graph: ['Conv', 'Relu', 'MaxPool', 'Conv', 'Relu', 'MaxPool', 'Conv', 'Relu', 'MaxPool', 'Flatten', 'Gemm', 'Relu', 'Gemm', 'Sigmoid']
File size: 44.7 MB


In [6]:
def decode(jpg):
    return cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)

def preprocess_bgr(img):
    img = cv2.resize(img, (224, 224))
    img = texture_segmentation(img)
    img = cv2.resize(img, (224, 224))
    img = (img - 127.5) / 255
    return img.astype(np.float32)[None, None]   # (1,1,224,224), same as .unsqueeze(1).float()

# Read JPEG bytes into memory up front so Google Drive I/O doesn't pollute the timing.
N_IMAGES = 32
jpg_bytes = []
try:
    start = int(len(labels_df) * 0.8)            # test split, same as TextureDataset(is_test=True)
    for name in labels_df['name'].values[start:start + N_IMAGES]:
        with open(os.path.join(image_path, f"{name}.jpg"), 'rb') as f:
            jpg_bytes.append(f.read())
    IMG_SOURCE = f"{len(jpg_bytes)} real test-split images"
except Exception as e:
    print("Dataset not available, using synthetic JPEGs:", type(e).__name__)
    rng = np.random.default_rng(0)
    jpg_bytes = [cv2.imencode('.jpg', rng.integers(0, 256, (866, 1154, 3), dtype=np.uint8))[1].tobytes()
                 for _ in range(N_IMAGES)]
    IMG_SOURCE = f"{N_IMAGES} synthetic 1154x866 noise JPEGs (decode time will be pessimistic)"

inputs = [np.ascontiguousarray(preprocess_bgr(decode(b))) for b in jpg_bytes]
print("Inputs:", IMG_SOURCE, "| tensor shape:", inputs[0].shape, inputs[0].dtype)

Inputs: 32 real test-split images | tensor shape: (1, 1, 224, 224) float32


In [7]:
def make_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = NUM_THREADS
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, sess_options=so, providers=['CPUExecutionProvider'])

sess        = make_session(ONNX_PATH)
sess_logits = make_session(LOGITS_PATH)
IN = sess.get_inputs()[0].name
print("ORT providers:", sess.get_providers())

def compare(torch_m, s, x):
    with torch.inference_mode():
        a = torch_m(torch.from_numpy(x)).numpy()
    b = s.run(None, {IN: x})[0]
    return a, b

prob_diffs, logit_diffs, agree, logits_seen = [], [], 0, []
for x in inputs:
    pa, pb = compare(cpu_model, sess, x)
    la, lb = compare(logits_model, sess_logits, x)
    prob_diffs.append(np.abs(pa - pb).max())
    logit_diffs.append(np.abs(la - lb).max())
    agree += int((pa > 0.5) == (pb > 0.5))
    logits_seen.append(la.item())

# dynamic batch axis check
xb = np.concatenate(inputs[:8])
pa, pb = compare(cpu_model, sess, xb)

print(f"Weights: {WEIGHTS} | inputs: {IMG_SOURCE}")
print(f"Max abs diff, probabilities (batch=1, {len(inputs)} imgs): {max(prob_diffs):.3e}")
print(f"Max abs diff, logits        (batch=1, {len(inputs)} imgs): {max(logit_diffs):.3e}")
print(f"Max abs diff, probabilities (batch=8, dynamic axis):      {np.abs(pa-pb).max():.3e}")
print(f"Prediction agreement PyTorch vs ORT: {agree}/{len(inputs)}")
print(f"Logit range seen: {min(logits_seen):.2f} to {max(logits_seen):.2f}")

ORT providers: ['CPUExecutionProvider']


/tmp/ipykernel_569/2989122056.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  agree += int((pa > 0.5) == (pb > 0.5))


Weights: random | inputs: 32 real test-split images
Max abs diff, probabilities (batch=1, 32 imgs): 2.384e-07
Max abs diff, logits        (batch=1, 32 imgs): 8.941e-07
Max abs diff, probabilities (batch=8, dynamic axis):      5.960e-08
Prediction agreement PyTorch vs ORT: 32/32
Logit range seen: -0.11 to 0.39


In [8]:
WARMUP, ROUNDS, RUNS_PER_ROUND = 20, 3, 100      # 300 timed runs each, interleaved in 3 rounds

def time_loop(fn, n):
    out = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); out.append((time.perf_counter() - t0) * 1000)
    return out

def bench(fns):
    res = {k: [] for k in fns}
    with torch.inference_mode():
        for f in fns.values():
            for _ in range(WARMUP): f()
        # interleave so a noisy-neighbor spike on Colab doesn't hit only one framework
        for _ in range(ROUNDS):
            for k, f in fns.items():
                res[k] += time_loop(f, RUNS_PER_ROUND)
    return {k: np.array(v) for k, v in res.items()}

def report(title, res, base, other):
    print(f"\n{title}")
    print(f"{'':16}{'mean':>9}{'p50':>9}{'p95':>9}{'std':>9}   n")
    for k, a in res.items():
        print(f"{k:16}{a.mean():9.2f}{np.percentile(a,50):9.2f}{np.percentile(a,95):9.2f}{a.std():9.2f}   {len(a)}")
    b, o = res[base], res[other]
    print(f"Speedup {base}/{other}: mean {b.mean()/o.mean():.2f}x | p50 {np.percentile(b,50)/np.percentile(o,50):.2f}x | p95 {np.percentile(b,95)/np.percentile(o,95):.2f}x")

# (a) model inference only, same input tensor for both
x_np = inputs[0]
x_t  = torch.from_numpy(x_np)                    # shares memory with x_np
model_only = bench({
    "PyTorch":     lambda: cpu_model(x_t),
    "ONNX Runtime": lambda: sess.run(None, {IN: x_np}),
})
report("(a) MODEL INFERENCE ONLY, batch=1, ms", model_only, "PyTorch", "ONNX Runtime")

# (b) per image end to end: JPEG decode + resize + Sobel + resize + normalize + inference
def e2e_pt():
    it = cycle(jpg_bytes)
    return lambda: cpu_model(torch.from_numpy(preprocess_bgr(decode(next(it)))))
def e2e_ort():
    it = cycle(jpg_bytes)
    return lambda: sess.run(None, {IN: preprocess_bgr(decode(next(it)))})
def prep_only():
    it = cycle(jpg_bytes)
    return lambda: preprocess_bgr(decode(next(it)))

e2e = bench({"PyTorch": e2e_pt(), "ONNX Runtime": e2e_ort(), "Preprocess only": prep_only()})
report("(b) END TO END PER IMAGE (decode + Sobel preprocessing + inference), batch=1, ms",
       e2e, "PyTorch", "ONNX Runtime")


(a) MODEL INFERENCE ONLY, batch=1, ms
                     mean      p50      p95      std   n
PyTorch             46.38    35.22   101.78    21.05   300
ONNX Runtime        17.42    16.84    19.93     2.15   300
Speedup PyTorch/ONNX Runtime: mean 2.66x | p50 2.09x | p95 5.11x

(b) END TO END PER IMAGE (decode + Sobel preprocessing + inference), batch=1, ms
                     mean      p50      p95      std   n
PyTorch             75.13    58.92   159.09    42.07   300
ONNX Runtime        49.83    44.13    73.40    24.13   300
Preprocess only     29.65    30.10    38.68     6.97   300
Speedup PyTorch/ONNX Runtime: mean 1.51x | p50 1.34x | p95 2.17x


In [9]:
from onnxruntime.quantization import quantize_dynamic, QuantType

INT8_PATH = '/content/model_int8.onnx'
try:
    # QUInt8 weights: ORT's CPU ConvInteger kernel only supports uint8, QInt8 often fails on Conv
    quantize_dynamic(ONNX_PATH, INT8_PATH, weight_type=QuantType.QUInt8)
    q = onnx.load(INT8_PATH)
    print("Int8 graph ops:", sorted({n.op_type for n in q.graph.node}))
    print(f"Size fp32 {os.path.getsize(ONNX_PATH)/1e6:.1f} MB -> int8 {os.path.getsize(INT8_PATH)/1e6:.1f} MB")

    sess_q = make_session(INT8_PATH)
    diffs, agree = [], 0
    for x in inputs:
        a = sess.run(None, {IN: x})[0]; b = sess_q.run(None, {IN: x})[0]
        diffs.append(np.abs(a - b).max()); agree += int((a > 0.5) == (b > 0.5))
    print(f"Int8 vs fp32 ORT: max abs prob diff {max(diffs):.3e}, mean {np.mean(diffs):.3e}, "
          f"prediction agreement {agree}/{len(inputs)} ({WEIGHTS} weights)")

    q_res = bench({
        "PyTorch fp32":  lambda: cpu_model(x_t),
        "ORT fp32":      lambda: sess.run(None, {IN: x_np}),
        "ORT int8":      lambda: sess_q.run(None, {IN: x_np}),
    })
    report("INT8 MODEL ONLY, batch=1, ms", q_res, "ORT fp32", "ORT int8")
    print(f"PyTorch fp32 / ORT int8 mean: {q_res['PyTorch fp32'].mean()/q_res['ORT int8'].mean():.2f}x")
except Exception as e:
    print("Int8 quantization failed:", type(e).__name__, str(e)[:300])

Int8 graph ops: ['Add', 'Cast', 'ConvInteger', 'DynamicQuantizeLinear', 'Flatten', 'MatMulInteger', 'MaxPool', 'Mul', 'Relu', 'Reshape', 'Sigmoid']
Size fp32 44.7 MB -> int8 11.2 MB


/tmp/ipykernel_569/1043522971.py:15: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  diffs.append(np.abs(a - b).max()); agree += int((a > 0.5) == (b > 0.5))


Int8 vs fp32 ORT: max abs prob diff 3.161e-03, mean 8.973e-04, prediction agreement 32/32 (random weights)

INT8 MODEL ONLY, batch=1, ms
                     mean      p50      p95      std   n
PyTorch fp32        45.26    35.54   102.24    20.57   300
ORT fp32            19.45    16.89    31.34     5.23   300
ORT int8            33.12    23.31    55.46    13.84   300
Speedup ORT fp32/ORT int8: mean 0.59x | p50 0.72x | p95 0.57x
PyTorch fp32 / ORT int8 mean: 1.37x
